# Assignment 2 - Experiment Tracking
Jugaad Singh Sohal (MDS202421)
## Model Training with MLflow

In this notebook we will train benchmark models on the SMS spam data and use MLflow to track experiments and register models. We train on both DVC data versions to demonstrate how data versioning affects model performance.

### Installing required packages

In [1]:
%pip install pandas numpy scikit-learn matplotlib mlflow


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import json
import subprocess
import time
import signal
import os
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import GridSearchCV
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings('ignore')

/home/jugaad/github/appliedmachinelearning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configure MLflow

Using SQLite backend to enable the model registry.

In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("sms-spam-classification")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('sms-spam-classification').name}")

2026/02/15 22:17:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas


2026/02/15 22:17:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables


2026/02/15 22:17:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types


2026/02/15 22:17:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints


2026/02/15 22:17:31 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults


...

2026/02/15 22:17:37 INFO alembic.runtime.migration: Running upgrade 2c33131f4dae -> d3e4f5a6b7c8, add display_name to endpoint_bindings


2026/02/15 22:17:37 INFO alembic.runtime.migration: Context impl SQLiteImpl.


2026/02/15 22:17:37 INFO alembic.runtime.migration: Will assume non-transactional DDL.


2026/02/15 22:17:37 INFO mlflow.tracking.fluent: Experiment with name 'sms-spam-classification' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db
Experiment: sms-spam-classification


### Training on Version 1 (random_state=42)

In [4]:
!git checkout v1 -- data/train.csv.dvc data/validation.csv.dvc data/test.csv.dvc
!dvc checkout

Building workspace index                              |0.00 [00:00,    ?entry/s]

Building workspace index                              |5.00 [00:00,  350entry/s]
Comparing indexes                                    |6.00 [00:00, 2.68kentry/s]
Applying changes                                      |3.00 [00:00,   220file/s]


M       data/test.csv
M       data/train.csv
M       data/validation.csv


In [5]:
train_v1 = pd.read_csv('data/train.csv')
val_v1 = pd.read_csv('data/validation.csv')
test_v1 = pd.read_csv('data/test.csv')

X_train_v1 = train_v1['message']
y_train_v1 = (train_v1['label'] == 'spam').astype(int)
X_val_v1 = val_v1['message']
y_val_v1 = (val_v1['label'] == 'spam').astype(int)
X_test_v1 = test_v1['message']
y_test_v1 = (test_v1['label'] == 'spam').astype(int)

print(f"V1 - Train: {len(train_v1)}, Val: {len(val_v1)}, Test: {len(test_v1)}")
print(f"V1 - Spam in train: {y_train_v1.sum()}, val: {y_val_v1.sum()}, test: {y_test_v1.sum()}")

V1 - Train: 3900, Val: 836, Test: 836
V1 - Spam in train: 523, val: 112, test: 112


In [6]:
tfidf_v1 = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_v1_tfidf = tfidf_v1.fit_transform(X_train_v1)
X_val_v1_tfidf = tfidf_v1.transform(X_val_v1)
X_test_v1_tfidf = tfidf_v1.transform(X_test_v1)
print(f"V1 TF-IDF shape: {X_train_v1_tfidf.shape}")

V1 TF-IDF shape: (3900, 5000)


In [7]:
lr_grid_v1 = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'],
     'solver': ['liblinear'], 'class_weight': [None, 'balanced']},
    cv=5, scoring='average_precision', n_jobs=-1
)
lr_grid_v1.fit(X_train_v1_tfidf, y_train_v1)

lr_v1 = lr_grid_v1.best_estimator_
val_aucpr_lr_v1 = average_precision_score(y_val_v1, lr_v1.predict_proba(X_val_v1_tfidf)[:, 1])
test_aucpr_lr_v1 = average_precision_score(y_test_v1, lr_v1.predict_proba(X_test_v1_tfidf)[:, 1])

with mlflow.start_run(run_name="logistic_regression_v1"):
    mlflow.log_params(lr_grid_v1.best_params_)
    mlflow.log_param("data_version", "v1")
    mlflow.log_param("max_features", 5000)
    mlflow.log_metric("cv_aucpr", lr_grid_v1.best_score_)
    mlflow.log_metric("val_aucpr", val_aucpr_lr_v1)
    mlflow.log_metric("test_aucpr", test_aucpr_lr_v1)
    mlflow.sklearn.log_model(lr_v1, "model",
                             registered_model_name="logistic_regression")

print(f"Best params: {lr_grid_v1.best_params_}")
print(f"CV AUCPR: {lr_grid_v1.best_score_:.4f}")
print(f"Val AUCPR: {val_aucpr_lr_v1:.4f}")
print(f"Test AUCPR: {test_aucpr_lr_v1:.4f}")

2026/02/15 22:17:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best params: {'C': 100, 'class_weight': None, 'penalty': 'l2', 'solver': 'liblinear'}
CV AUCPR: 0.9771
Val AUCPR: 0.9764
Test AUCPR: 0.9642


Successfully registered model 'logistic_regression'.
Created version '1' of model 'logistic_regression'.


In [8]:
nb_grid_v1 = GridSearchCV(
    MultinomialNB(),
    {'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0], 'fit_prior': [True, False]},
    cv=5, scoring='average_precision', n_jobs=-1
)
nb_grid_v1.fit(X_train_v1_tfidf, y_train_v1)

nb_v1 = nb_grid_v1.best_estimator_
val_aucpr_nb_v1 = average_precision_score(y_val_v1, nb_v1.predict_proba(X_val_v1_tfidf)[:, 1])
test_aucpr_nb_v1 = average_precision_score(y_test_v1, nb_v1.predict_proba(X_test_v1_tfidf)[:, 1])

with mlflow.start_run(run_name="naive_bayes_v1"):
    mlflow.log_params(nb_grid_v1.best_params_)
    mlflow.log_param("data_version", "v1")
    mlflow.log_param("max_features", 5000)
    mlflow.log_metric("cv_aucpr", nb_grid_v1.best_score_)
    mlflow.log_metric("val_aucpr", val_aucpr_nb_v1)
    mlflow.log_metric("test_aucpr", test_aucpr_nb_v1)
    mlflow.sklearn.log_model(nb_v1, "model",
                             registered_model_name="naive_bayes")

print(f"Best params: {nb_grid_v1.best_params_}")
print(f"CV AUCPR: {nb_grid_v1.best_score_:.4f}")
print(f"Val AUCPR: {val_aucpr_nb_v1:.4f}")
print(f"Test AUCPR: {test_aucpr_nb_v1:.4f}")

2026/02/15 22:17:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best params: {'alpha': 0.1, 'fit_prior': True}
CV AUCPR: 0.9783
Val AUCPR: 0.9767
Test AUCPR: 0.9708


Successfully registered model 'naive_bayes'.
Created version '1' of model 'naive_bayes'.


In [9]:
svm_grid_v1 = GridSearchCV(
    LinearSVC(random_state=42, max_iter=5000),
    {'C': [0.01, 0.1, 1, 10, 100], 'loss': ['hinge', 'squared_hinge'],
     'class_weight': [None, 'balanced']},
    cv=5, scoring='average_precision', n_jobs=-1
)
svm_grid_v1.fit(X_train_v1_tfidf, y_train_v1)

svm_v1 = svm_grid_v1.best_estimator_
val_aucpr_svm_v1 = average_precision_score(y_val_v1, svm_v1.decision_function(X_val_v1_tfidf))
test_aucpr_svm_v1 = average_precision_score(y_test_v1, svm_v1.decision_function(X_test_v1_tfidf))

with mlflow.start_run(run_name="linear_svc_v1"):
    mlflow.log_params(svm_grid_v1.best_params_)
    mlflow.log_param("data_version", "v1")
    mlflow.log_param("max_features", 5000)
    mlflow.log_metric("cv_aucpr", svm_grid_v1.best_score_)
    mlflow.log_metric("val_aucpr", val_aucpr_svm_v1)
    mlflow.log_metric("test_aucpr", test_aucpr_svm_v1)
    mlflow.sklearn.log_model(svm_v1, "model",
                             registered_model_name="linear_svc")

print(f"Best params: {svm_grid_v1.best_params_}")
print(f"CV AUCPR: {svm_grid_v1.best_score_:.4f}")
print(f"Val AUCPR: {val_aucpr_svm_v1:.4f}")
print(f"Test AUCPR: {test_aucpr_svm_v1:.4f}")

2026/02/15 22:17:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best params: {'C': 1, 'class_weight': None, 'loss': 'hinge'}
CV AUCPR: 0.9784
Val AUCPR: 0.9774
Test AUCPR: 0.9672


Successfully registered model 'linear_svc'.
Created version '1' of model 'linear_svc'.


### Training on Version 2 (random_state=123)

In [10]:
!git checkout v2 -- data/train.csv.dvc data/validation.csv.dvc data/test.csv.dvc
!dvc checkout

Building workspace index                              |5.00 [00:00,  292entry/s]
Comparing indexes                                     |0.00 [00:00,    ?entry/s]

Comparing indexes                                    |6.00 [00:00, 2.12kentry/s]
Applying changes                                      |3.00 [00:00,   212file/s]


M       data/test.csv
M       data/train.csv
M       data/validation.csv


In [11]:
train_v2 = pd.read_csv('data/train.csv')
val_v2 = pd.read_csv('data/validation.csv')
test_v2 = pd.read_csv('data/test.csv')

X_train_v2 = train_v2['message']
y_train_v2 = (train_v2['label'] == 'spam').astype(int)
X_val_v2 = val_v2['message']
y_val_v2 = (val_v2['label'] == 'spam').astype(int)
X_test_v2 = test_v2['message']
y_test_v2 = (test_v2['label'] == 'spam').astype(int)

print(f"V2 - Train: {len(train_v2)}, Val: {len(val_v2)}, Test: {len(test_v2)}")
print(f"V2 - Spam in train: {y_train_v2.sum()}, val: {y_val_v2.sum()}, test: {y_test_v2.sum()}")

V2 - Train: 3900, Val: 836, Test: 836
V2 - Spam in train: 523, val: 112, test: 112


In [12]:
tfidf_v2 = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_v2_tfidf = tfidf_v2.fit_transform(X_train_v2)
X_val_v2_tfidf = tfidf_v2.transform(X_val_v2)
X_test_v2_tfidf = tfidf_v2.transform(X_test_v2)
print(f"V2 TF-IDF shape: {X_train_v2_tfidf.shape}")

V2 TF-IDF shape: (3900, 5000)


In [13]:
lr_grid_v2 = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'],
     'solver': ['liblinear'], 'class_weight': [None, 'balanced']},
    cv=5, scoring='average_precision', n_jobs=-1
)
lr_grid_v2.fit(X_train_v2_tfidf, y_train_v2)

lr_v2 = lr_grid_v2.best_estimator_
val_aucpr_lr_v2 = average_precision_score(y_val_v2, lr_v2.predict_proba(X_val_v2_tfidf)[:, 1])
test_aucpr_lr_v2 = average_precision_score(y_test_v2, lr_v2.predict_proba(X_test_v2_tfidf)[:, 1])

with mlflow.start_run(run_name="logistic_regression_v2"):
    mlflow.log_params(lr_grid_v2.best_params_)
    mlflow.log_param("data_version", "v2")
    mlflow.log_param("max_features", 5000)
    mlflow.log_metric("cv_aucpr", lr_grid_v2.best_score_)
    mlflow.log_metric("val_aucpr", val_aucpr_lr_v2)
    mlflow.log_metric("test_aucpr", test_aucpr_lr_v2)
    mlflow.sklearn.log_model(lr_v2, "model",
                             registered_model_name="logistic_regression")

print(f"Best params: {lr_grid_v2.best_params_}")
print(f"CV AUCPR: {lr_grid_v2.best_score_:.4f}")
print(f"Val AUCPR: {val_aucpr_lr_v2:.4f}")
print(f"Test AUCPR: {test_aucpr_lr_v2:.4f}")

2026/02/15 22:18:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best params: {'C': 100, 'class_weight': None, 'penalty': 'l2', 'solver': 'liblinear'}
CV AUCPR: 0.9734
Val AUCPR: 0.9638
Test AUCPR: 0.9737


Registered model 'logistic_regression' already exists. Creating a new version of this model...
Created version '2' of model 'logistic_regression'.


In [14]:
nb_grid_v2 = GridSearchCV(
    MultinomialNB(),
    {'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0], 'fit_prior': [True, False]},
    cv=5, scoring='average_precision', n_jobs=-1
)
nb_grid_v2.fit(X_train_v2_tfidf, y_train_v2)

nb_v2 = nb_grid_v2.best_estimator_
val_aucpr_nb_v2 = average_precision_score(y_val_v2, nb_v2.predict_proba(X_val_v2_tfidf)[:, 1])
test_aucpr_nb_v2 = average_precision_score(y_test_v2, nb_v2.predict_proba(X_test_v2_tfidf)[:, 1])

with mlflow.start_run(run_name="naive_bayes_v2"):
    mlflow.log_params(nb_grid_v2.best_params_)
    mlflow.log_param("data_version", "v2")
    mlflow.log_param("max_features", 5000)
    mlflow.log_metric("cv_aucpr", nb_grid_v2.best_score_)
    mlflow.log_metric("val_aucpr", val_aucpr_nb_v2)
    mlflow.log_metric("test_aucpr", test_aucpr_nb_v2)
    mlflow.sklearn.log_model(nb_v2, "model",
                             registered_model_name="naive_bayes")

print(f"Best params: {nb_grid_v2.best_params_}")
print(f"CV AUCPR: {nb_grid_v2.best_score_:.4f}")
print(f"Val AUCPR: {val_aucpr_nb_v2:.4f}")
print(f"Test AUCPR: {test_aucpr_nb_v2:.4f}")

2026/02/15 22:18:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best params: {'alpha': 0.1, 'fit_prior': True}
CV AUCPR: 0.9765
Val AUCPR: 0.9774
Test AUCPR: 0.9706


Registered model 'naive_bayes' already exists. Creating a new version of this model...
Created version '2' of model 'naive_bayes'.


In [15]:
svm_grid_v2 = GridSearchCV(
    LinearSVC(random_state=42, max_iter=5000),
    {'C': [0.01, 0.1, 1, 10, 100], 'loss': ['hinge', 'squared_hinge'],
     'class_weight': [None, 'balanced']},
    cv=5, scoring='average_precision', n_jobs=-1
)
svm_grid_v2.fit(X_train_v2_tfidf, y_train_v2)

svm_v2 = svm_grid_v2.best_estimator_
val_aucpr_svm_v2 = average_precision_score(y_val_v2, svm_v2.decision_function(X_val_v2_tfidf))
test_aucpr_svm_v2 = average_precision_score(y_test_v2, svm_v2.decision_function(X_test_v2_tfidf))

with mlflow.start_run(run_name="linear_svc_v2"):
    mlflow.log_params(svm_grid_v2.best_params_)
    mlflow.log_param("data_version", "v2")
    mlflow.log_param("max_features", 5000)
    mlflow.log_metric("cv_aucpr", svm_grid_v2.best_score_)
    mlflow.log_metric("val_aucpr", val_aucpr_svm_v2)
    mlflow.log_metric("test_aucpr", test_aucpr_svm_v2)
    mlflow.sklearn.log_model(svm_v2, "model",
                             registered_model_name="linear_svc")

print(f"Best params: {svm_grid_v2.best_params_}")
print(f"CV AUCPR: {svm_grid_v2.best_score_:.4f}")
print(f"Val AUCPR: {val_aucpr_svm_v2:.4f}")
print(f"Test AUCPR: {test_aucpr_svm_v2:.4f}")

2026/02/15 22:18:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best params: {'C': 1, 'class_weight': None, 'loss': 'squared_hinge'}
CV AUCPR: 0.9761
Val AUCPR: 0.9641
Test AUCPR: 0.9748


Registered model 'linear_svc' already exists. Creating a new version of this model...
Created version '2' of model 'linear_svc'.
